# 03 · CKM Biomarker Integration
## NHANES 2017–March 2020 Women's CKM Phenotyping Project

---

**Author:** Alexandra Velez, MD  
**Input:** `data/processed/reproductive_features.csv` — 1,603 women aged 20–44  
**Output:** `data/processed/ckm_features.csv`  
**Last updated:** 2026

---

### Purpose of this notebook

This notebook integrates cardiometabolic-kidney-metabolic (CKM) biomarkers 
with the reproductive health features engineered in notebook 02. The 
integration is performed by linking five NHANES biomarker modules to the 
reproductive feature matrix via SEQN — the universal participant identifier 
across all NHANES files.

The output is a complete CKM feature matrix combining reproductive history 
with metabolic biomarkers — the dataset that will drive the unsupervised 
clustering analysis in notebook 05.

---

### Clinical rationale

Reproductive features alone are insufficient to identify cardiometabolic 
phenotypes — they provide the exposure history but not the current 
metabolic state. CKM biomarkers provide the outcome side of the equation: 
how a woman's cardiometabolic system looks today, at ages 20–44, after 
whatever reproductive exposures she has experienced.

The central question this integration enables:

> Do women with adverse pregnancy outcomes — particularly gestational 
> diabetes — already show distinct early cardiometabolic biomarker 
> profiles at ages 20–44, before overt cardiovascular disease has 
> had time to develop?

---

### Two-stage clustering design

As established in notebook 02, a two-stage analysis is adopted due to 
the NHANES fasting subsample design:

**Stage 1 — Primary clustering (n=1,603):**
Reproductive features + non-fasting CKM biomarkers. Preserves the full 
analytical sample.

**Stage 2 — Fasting subsample characterization (n≈745):**
Triglycerides and LDL examined within Stage 1 clusters. Adds lipid depth 
without compromising sample size.

---

### Modules integrated in this notebook

| Module | File | Key variables | Coverage |
|---|---|---|---|
| Reproductive features | `reproductive_features.csv` | 19 features | notebook 02 |
| Glycohemoglobin | P_GHB.xpt | HbA1c | ~95% |
| Body measures | P_BMX.xpt | BMI, waist | ~97-99% |
| Blood pressure | P_BPXO.xpt | Systolic, diastolic BP | ~91% |
| Biochemistry | P_BIOPRO.xpt | Creatinine, glucose, ALT | ~93% |
| HDL cholesterol | P_HDL.xpt | HDL | ~94% |
| Triglycerides & LDL | P_TRIGLY.xpt | Triglycerides, LDL | ~47% |
| Prescription medications | P_RXQ_RX.xpt | CKM-relevant drug flags | TBD |

---

### Notebook inputs and outputs

| File | Description |
|---|---|
| `data/processed/reproductive_features.csv` | Input — from notebook 02 |
| `data/processed/ckm_features.csv` | Output — complete CKM feature matrix |
| `data/processed/feature_classification.csv` | Reference — clustering feature list |

---
## Section 1 · Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import os

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
INPUT_PATH  = Path('../data/processed/reproductive_features.csv')
CLASS_PATH  = Path('../data/processed/feature_classification.csv')
RAW_DIR     = Path('../data/raw')
OUTPUT_PATH = Path('../data/processed/ckm_features.csv')
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Raw module paths ───────────────────────────────────────────────────────
MODULE_PATHS = {
    'P_GHB':    RAW_DIR / 'P_GHB.xpt',
    'P_BMX':    RAW_DIR / 'P_BMX.xpt',
    'P_BPXO':   RAW_DIR / 'P_BPXO.xpt',
    'P_BIOPRO': RAW_DIR / 'P_BIOPRO.xpt',
    'P_HDL':    RAW_DIR / 'P_HDL.xpt',
    'P_TRIGLY': RAW_DIR / 'P_TRIGLY.xpt',
    'P_RXQ_RX': RAW_DIR / 'P_RXQ_RX.xpt',
}

# ── Plot style ─────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Setup complete.')
print(f'  Input:  {INPUT_PATH}')
print(f'  Output: {OUTPUT_PATH}')
print(f'\nRaw modules:')
for name, path in MODULE_PATHS.items():
    exists = '✓' if path.exists() else '⚠️  NOT FOUND'
    print(f'  {exists}  {name}  ({path.name})')

Setup complete.
  Input:  ../data/processed/reproductive_features.csv
  Output: ../data/processed/ckm_features.csv

Raw modules:
  ✓  P_GHB  (P_GHB.xpt)
  ✓  P_BMX  (P_BMX.xpt)
  ✓  P_BPXO  (P_BPXO.xpt)
  ✓  P_BIOPRO  (P_BIOPRO.xpt)
  ✓  P_HDL  (P_HDL.xpt)
  ✓  P_TRIGLY  (P_TRIGLY.xpt)
  ✓  P_RXQ_RX  (P_RXQ_RX.xpt)


---
## Section 2 · Load Reproductive Features & SAS Artifact Scan

The reproductive feature matrix from notebook 02 is loaded first. 
Then all six CKM biomarker modules are loaded and scanned for SAS XPT 
floating point artifacts before any variable selection or merging takes 
place. As established in notebook 01, this scan is mandatory for any 
SAS XPT file and must run before any analysis.

In [2]:
# ── Load reproductive feature matrix ──────────────────────────────────────
df_repro = pd.read_csv(INPUT_PATH, index_col='SEQN')

assert df_repro.shape == (1603, 26), \
    f'Unexpected shape {df_repro.shape} — expected (1603, 26)'
assert df_repro.index.name == 'SEQN', \
    'SEQN is not the index'

print(f'✓ Reproductive feature matrix loaded: {df_repro.shape}')
print(f'  Age range: {df_repro["RIDAGEYR"].min():.0f}–'
      f'{df_repro["RIDAGEYR"].max():.0f} years')

# Load feature classification reference
df_class = pd.read_csv(CLASS_PATH)
clustering_features = df_class.loc[
    df_class['clustering'] == True, 'feature'
].tolist()
print(f'\n✓ Feature classification loaded')
print(f'  Clustering features: {clustering_features}')

✓ Reproductive feature matrix loaded: (1603, 26)
  Age range: 20–44 years

✓ Feature classification loaded
  Clustering features: ['Age_Menarche', 'Parity', 'Pregnancy_Loss', 'APO_Score']


In [3]:
# ── SAS XPT artifact scan — all CKM modules ───────────────────────────────
# Mandatory first step before any data processing
# scan_sas_artifacts() defined here for notebook independence

EPSILON = 1e-10

def scan_sas_artifacts(df, epsilon=EPSILON):
    """
    Scan a dataframe for SAS XPT floating point artifacts.
    True zeros sometimes exported as ~5.397605e-79 in SAS XPT files.
    Returns dict {column: count} for affected columns.
    """
    affected = {}
    for col in df.select_dtypes(include='number').columns:
        mask = (
            df[col].notna() &
            (df[col].abs() < epsilon) &
            (df[col] != 0.0)
        )
        n = mask.sum()
        if n > 0:
            affected[col] = n
    return affected

# ── Load and scan all modules ──────────────────────────────────────────────
modules = {}
total_artifacts = 0

print('=== SAS XPT Artifact Scan — CKM Modules ===\n')
print(f'  {"Module":<12} {"Rows":>7} {"Cols":>6} {"Artifacts":>10}')
print(f'  {"─"*12} {"─"*7} {"─"*6} {"─"*10}')

for name, path in MODULE_PATHS.items():
    df = pd.read_sas(path, format='xport', encoding='utf-8')
    df = df.set_index('SEQN')

    # Scan for artifacts
    artifacts = scan_sas_artifacts(df)
    n_artifacts = sum(artifacts.values())
    total_artifacts += n_artifacts

    # Fix if needed
    if artifacts:
        for col, n in artifacts.items():
            df[col] = np.where(
                df[col].notna() & (df[col].abs() < EPSILON),
                0.0,
                df[col]
            )
        status = f'{n_artifacts:,} fixed'
    else:
        status = '✓ clean'

    modules[name] = df
    print(f'  {name:<12} {len(df):>7,} {len(df.columns):>6,} {status:>10}')

print(f'\n  Total artifact values fixed: {total_artifacts:,}')
print(f'  All modules artifact-free after fix ✓' 
      if total_artifacts >= 0 else '')

# Final validation
for name, df in modules.items():
    remaining = scan_sas_artifacts(df)
    assert len(remaining) == 0, \
        f'Artifacts remain in {name}: {remaining}'

print(f'\n✓ All CKM modules loaded and artifact-free')
print(f'  Modules ready: {list(modules.keys())}')

=== SAS XPT Artifact Scan — CKM Modules ===

  Module          Rows   Cols  Artifacts
  ──────────── ─────── ────── ──────────
  P_GHB         10,409      1    ✓ clean
  P_BMX         14,300     21    ✓ clean
  P_BPXO        11,656     11    ✓ clean
  P_BIOPRO      10,409     40 28,417 fixed
  P_HDL         12,198      2    ✓ clean
  P_TRIGLY       5,090      9  614 fixed
  P_RXQ_RX      32,962     12    ✓ clean

  Total artifact values fixed: 29,031
  All modules artifact-free after fix ✓

✓ All CKM modules loaded and artifact-free
  Modules ready: ['P_GHB', 'P_BMX', 'P_BPXO', 'P_BIOPRO', 'P_HDL', 'P_TRIGLY', 'P_RXQ_RX']


---
## Section 3 · CKM Variable Selection

Key CKM biomarker variables are selected from each module. Only variables 
needed for feature engineering are extracted — not all columns from each 
module. Variable names and clinical units are documented here for 
reference throughout the notebook.

### Variable selection rationale

Variables are selected based on three criteria:
1. **Clinical relevance** — direct relevance to CKM risk assessment
2. **Coverage** — confirmed above 70% in the analytical sample 
   (notebook 02 coverage check)
3. **Non-redundancy** — where multiple variables measure the same 
   construct, the most clinically standard is selected

### Fasting vs non-fasting distinction

A critical distinction in NHANES laboratory data is whether a variable 
requires a fasting blood draw. Non-fasting variables are available for 
all examined participants. Fasting variables are only available for the 
morning session subsample (~50% of participants).

**Non-fasting (Stage 1 — full sample):**
HbA1c, BMI, waist circumference, blood pressure, HDL, creatinine, 
glucose, ALT

**Fasting (Stage 2 — subsample only):**
Triglycerides, LDL cholesterol

In [4]:
# ── CKM variable selection ─────────────────────────────────────────────────
# Variables selected from each module with clinical labels
# All confirmed available in notebook 02 coverage check

CKM_VARS = {
    # ── P_GHB — Glycohemoglobin ───────────────────────────────────────────
    'LBXGH':    ('P_GHB',    'HbA1c (%)',
                 'Non-fasting — primary glucose dysregulation marker'),

    # ── P_BMX — Body Measures ─────────────────────────────────────────────
    'BMXBMI':   ('P_BMX',    'BMI (kg/m²)',
                 'Non-fasting — adiposity marker'),
    'BMXWAIST': ('P_BMX',    'Waist circumference (cm)',
                 'Non-fasting — central adiposity marker'),

    # ── P_BPXO — Blood Pressure ───────────────────────────────────────────
    'BPXOSY1':  ('P_BPXO',   'Systolic BP reading 1 (mmHg)',
                 'Non-fasting'),
    'BPXOSY2':  ('P_BPXO',   'Systolic BP reading 2 (mmHg)',
                 'Non-fasting'),
    'BPXOSY3':  ('P_BPXO',   'Systolic BP reading 3 (mmHg)',
                 'Non-fasting'),
    'BPXODI1':  ('P_BPXO',   'Diastolic BP reading 1 (mmHg)',
                 'Non-fasting'),
    'BPXODI2':  ('P_BPXO',   'Diastolic BP reading 2 (mmHg)',
                 'Non-fasting'),
    'BPXODI3':  ('P_BPXO',   'Diastolic BP reading 3 (mmHg)',
                 'Non-fasting'),

    # ── P_BIOPRO — Biochemistry ───────────────────────────────────────────
    'LBXSCR':   ('P_BIOPRO', 'Serum creatinine (mg/dL)',
                 'Non-fasting — used to calculate eGFR'),
    'LBXSGL':   ('P_BIOPRO', 'Glucose (mg/dL)',
                 'Non-fasting — random glucose'),
    'LBXSATSI': ('P_BIOPRO', 'ALT (U/L)',
                 'Non-fasting — liver function marker'),

    # ── P_HDL — HDL Cholesterol ───────────────────────────────────────────
    'LBDHDD':   ('P_HDL',    'HDL cholesterol (mg/dL)',
                 'Non-fasting — cardioprotective lipid'),

    # ── P_TRIGLY — Fasting Lipids ─────────────────────────────────────────
    'LBXTR':    ('P_TRIGLY', 'Triglycerides (mg/dL)',
                 'Fasting — Stage 2 only'),
    'LBDLDL':   ('P_TRIGLY', 'LDL cholesterol (mg/dL)',
                 'Fasting — Stage 2 only'),
    'WTSAFPRP': ('P_TRIGLY', 'Fasting subsample weight',
                 'Survey weight for fasting subsample analyses'),
}

# ── Extract selected variables from each module ────────────────────────────
print('=== Variable Selection from CKM Modules ===\n')
print(f'  {"Variable":<12} {"Module":<10} {"Found":>6}  {"Label"}')
print(f'  {"─"*12} {"─"*10} {"─"*6}  {"─"*40}')

extracted = {}
missing   = []

for var, (module, label, note) in CKM_VARS.items():
    df_mod = modules[module]
    if var in df_mod.columns:
        extracted[var] = df_mod[var]
        print(f'  {var:<12} {module:<10} {"✓":>6}  {label}')
    else:
        missing.append((var, module, label))
        print(f'  {var:<12} {module:<10} {"⚠️":>6}  {label} — NOT FOUND')

if missing:
    print(f'\n⚠️  {len(missing)} variables not found — check variable names')
else:
    print(f'\n✓ All {len(CKM_VARS)} variables found in their respective modules')

=== Variable Selection from CKM Modules ===

  Variable     Module      Found  Label
  ──────────── ────────── ──────  ────────────────────────────────────────
  LBXGH        P_GHB           ✓  HbA1c (%)
  BMXBMI       P_BMX           ✓  BMI (kg/m²)
  BMXWAIST     P_BMX           ✓  Waist circumference (cm)
  BPXOSY1      P_BPXO          ✓  Systolic BP reading 1 (mmHg)
  BPXOSY2      P_BPXO          ✓  Systolic BP reading 2 (mmHg)
  BPXOSY3      P_BPXO          ✓  Systolic BP reading 3 (mmHg)
  BPXODI1      P_BPXO          ✓  Diastolic BP reading 1 (mmHg)
  BPXODI2      P_BPXO          ✓  Diastolic BP reading 2 (mmHg)
  BPXODI3      P_BPXO          ✓  Diastolic BP reading 3 (mmHg)
  LBXSCR       P_BIOPRO        ✓  Serum creatinine (mg/dL)
  LBXSGL       P_BIOPRO        ✓  Glucose (mg/dL)
  LBXSATSI     P_BIOPRO        ✓  ALT (U/L)
  LBDHDD       P_HDL           ✓  HDL cholesterol (mg/dL)
  LBXTR        P_TRIGLY        ✓  Triglycerides (mg/dL)
  LBDLDL       P_TRIGLY        ✓  LDL chole

---
## Section 4 · Module Integration

All six CKM modules are merged onto the reproductive feature matrix 
using SEQN as the linking key. A left join is used for each merge — 
this preserves all 1,603 women in the analytical sample regardless of 
whether they have data in a given module. Women absent from a module 
receive NaN for that module's variables.

All merges are validated immediately after joining to confirm the 
row count remains 1,603 throughout.

In [5]:
# ── Merge CKM variables onto reproductive feature matrix ───────────────────
# Left join on SEQN index — preserves all 1,603 analytical sample women
# Women absent from a module receive NaN for that module's variables
# Left join is the only correct choice here —
# df_ckm is the analytical anchor (1,603 women, carefully defined
# in notebook 01). A right or inner join would expand or contract
# the sample by including non-analytical participants from the
# CKM modules or dropping women without biomarker data.
# NaN values from missing biomarker data are handled downstream
# during feature engineering and imputation decisions.

df_ckm = df_repro.copy()

print('=== Module Integration ===\n')
print(f'  Starting shape: {df_ckm.shape}')
print(f'  {"─"*50}')

# Build a single dataframe of all extracted CKM variables
# Group by module to perform one join per module
module_dfs = {}
for var, (module, label, note) in CKM_VARS.items():
    if module not in module_dfs:
        module_dfs[module] = []
    module_dfs[module].append(var)

for module, var_list in module_dfs.items():
    df_mod     = modules[module][var_list]
    n_before   = len(df_ckm)
    df_ckm     = df_ckm.join(df_mod, how='left')
    n_after    = len(df_ckm)

    # Count how many analytical sample women have data from this module
    n_with_data = df_ckm[var_list[0]].notna().sum()
    pct         = n_with_data / len(df_ckm) * 100

    assert n_after == n_before, \
        f'Row count changed after merging {module}: {n_before} → {n_after}'

    print(f'  ✓ {module:<12} merged  '
          f'({n_with_data:,} of 1,603 women have data — {pct:.1f}%)')

print(f'  {"─"*50}')
print(f'  Final shape: {df_ckm.shape}')
print(f'  ({df_ckm.shape[1] - 25} CKM variables + 25 reproductive/demo)')

# ── Validate merge ─────────────────────────────────────────────────────────
assert len(df_ckm) == 1603, \
    f'Row count changed during merge: {len(df_ckm)}'
assert df_ckm.index.name == 'SEQN', \
    'SEQN lost as index during merge'
assert all(var in df_ckm.columns for var in CKM_VARS.keys()), \
    'Some CKM variables missing after merge'

print(f'\n✓ All merge assertions passed')
print(f'✓ 1,603 women retained throughout — left join confirmed')

=== Module Integration ===

  Starting shape: (1603, 26)
  ──────────────────────────────────────────────────
  ✓ P_GHB        merged  (1,522 of 1,603 women have data — 94.9%)
  ✓ P_BMX        merged  (1,596 of 1,603 women have data — 99.6%)
  ✓ P_BPXO       merged  (1,451 of 1,603 women have data — 90.5%)
  ✓ P_BIOPRO     merged  (1,499 of 1,603 women have data — 93.5%)
  ✓ P_HDL        merged  (1,503 of 1,603 women have data — 93.8%)
  ✓ P_TRIGLY     merged  (745 of 1,603 women have data — 46.5%)
  ──────────────────────────────────────────────────
  Final shape: (1603, 42)
  (17 CKM variables + 25 reproductive/demo)

✓ All merge assertions passed
✓ 1,603 women retained throughout — left join confirmed


---
## Section 5 · CKM Feature Engineering

Raw biomarker variables require transformation before clustering. This 
section derives CKM features from the merged dataset, applying clinical 
cleaning rules and validated equations where appropriate.

**Features engineered in this section:**

- **Mean_SBP** — mean systolic blood pressure from up to 3 oscillometric 
  readings. NHANES protocol collects 3 readings — the mean is more 
  reliable than any single reading and is the standard approach in 
  epidemiological analyses.

- **Mean_DBP** — mean diastolic blood pressure, same approach as Mean_SBP.

- **eGFR** — estimated glomerular filtration rate calculated from serum 
  creatinine using the CKD-EPI 2021 equation (race-free, sex-specific). 
  The 2021 equation is used rather than the older 2009 equation because 
  it removes race as a variable — consistent with current clinical 
  guidelines and methodologically appropriate for a diverse population 
  sample.

- **HbA1c, BMI, Waist, HDL, Glucose, ALT, Triglycerides, LDL** — used 
  directly from raw variables after clinical range validation.

r
### CKD-EPI 2021 equation

The race-free CKD-EPI 2021 equation for eGFR:

```
eGFR = 142 × min(Scr/κ, 1)^α × max(Scr/κ, 1)^(-1.200) 
       × 0.9938^Age

Where:
  κ = 0.7 (females)
  α = -0.241 (females)
  Scr = serum creatinine (mg/dL)
```

In [6]:
# ── CKM feature engineering ────────────────────────────────────────────────
ckm_features = pd.DataFrame(index=df_ckm.index)

# ── Feature 1 & 2: Mean_SBP and Mean_DBP ──────────────────────────────────
# Average of up to 3 oscillometric readings
# NaN readings excluded from mean — participant retains valid mean
# if at least 1 reading is available

sbp_cols = ['BPXOSY1', 'BPXOSY2', 'BPXOSY3']
dbp_cols = ['BPXODI1', 'BPXODI2', 'BPXODI3']

ckm_features['Mean_SBP'] = df_ckm[sbp_cols].mean(axis=1, skipna=True)
ckm_features['Mean_DBP'] = df_ckm[dbp_cols].mean(axis=1, skipna=True)

# Set NaN where all readings are missing
ckm_features.loc[df_ckm[sbp_cols].isna().all(axis=1), 'Mean_SBP'] = np.nan
ckm_features.loc[df_ckm[dbp_cols].isna().all(axis=1), 'Mean_DBP'] = np.nan

print('Blood pressure features:')
print(f'  Mean_SBP valid: {ckm_features["Mean_SBP"].notna().sum():,}')
print(f'  Mean_DBP valid: {ckm_features["Mean_DBP"].notna().sum():,}')

# How many readings contributed to mean
n_readings = df_ckm[sbp_cols].notna().sum(axis=1)
print(f'  Readings per participant:')
for n in [1, 2, 3]:
    print(f'    {n} reading(s): {(n_readings == n).sum():,}')
print(f'    0 readings:   {(n_readings == 0).sum():,}')

# ── Feature 3: eGFR ───────────────────────────────────────────────────────
# CKD-EPI 2021 race-free equation — females only
# κ = 0.7, α = -0.241 for females
# All analytical sample participants are female (P_RHQ module)

kappa = 0.7
alpha = -0.241

scr   = df_ckm['LBXSCR']
age   = df_ckm['RIDAGEYR']

# Calculate eGFR
scr_ratio = scr / kappa
egfr = (
    142 *
    np.where(scr_ratio < 1, scr_ratio ** alpha, 1.0) *
    np.where(scr_ratio > 1, scr_ratio ** -1.200, 1.0) *
    (0.9938 ** age)
)

# Set NaN where creatinine is missing
egfr = np.where(scr.isna(), np.nan, egfr)
ckm_features['eGFR'] = egfr

print(f'\neGFR (CKD-EPI 2021):')
print(f'  Valid:  {ckm_features["eGFR"].notna().sum():,}')
print(f'  Mean:   {ckm_features["eGFR"].mean():.1f} mL/min/1.73m²')
print(f'  Range:  {ckm_features["eGFR"].min():.1f}–'
      f'{ckm_features["eGFR"].max():.1f} mL/min/1.73m²')

# ── Features 4–11: Direct biomarkers ──────────────────────────────────────
direct_vars = {
    'HbA1c':         'LBXGH',
    'BMI':           'BMXBMI',
    'Waist':         'BMXWAIST',
    'HDL':           'LBDHDD',
    'Glucose':       'LBXSGL',
    'ALT':           'LBXSATSI',
    'Triglycerides': 'LBXTR',
    'LDL':           'LBDLDL',
}

for feat, var in direct_vars.items():
    ckm_features[feat] = df_ckm[var]

# ── Also carry forward fasting weight ─────────────────────────────────────
ckm_features['WTSAFPRP'] = df_ckm['WTSAFPRP']

print(f'\nDirect biomarkers assigned:')
for feat, var in direct_vars.items():
    valid = ckm_features[feat].notna().sum()
    pct   = valid / len(ckm_features) * 100
    print(f'  {feat:<15} {valid:>6,}  ({pct:.1f}%)')

# ── Clinical range validation ──────────────────────────────────────────────
clinical_ranges = {
    'HbA1c':         (3.0,   20.0),
    'BMI':           (10.0,  100.0),  # raised from 80 — extreme obesity real
    'Waist':         (40.0,  200.0),
    'Mean_SBP':      (60.0,  250.0),
    'Mean_DBP':      (30.0,  150.0),
    'eGFR':          (3.0,   200.0),  # lowered from 5 — ESKD real
    'HDL':           (10.0,  200.0),  # raised from 150 — hyperalpha real
    'Glucose':       (30.0,  600.0),
    'ALT':           (1.0,   500.0),
    'Triglycerides': (20.0,  2000.0), # unchanged — <20 implausible
    'LDL':           (10.0,  600.0),
}

print(f'\nClinical range validation:')
total_flagged = 0
for feat, (lo, hi) in clinical_ranges.items():
    if feat not in ckm_features.columns:
        continue
    col          = ckm_features[feat]
    out_of_range = col.notna() & ((col < lo) | (col > hi))
    n_flagged    = out_of_range.sum()
    total_flagged += n_flagged
    if n_flagged > 0:
        ckm_features.loc[out_of_range, feat] = np.nan
        print(f'  ⚠️  {feat:<15} {n_flagged:>4,} values outside '
              f'[{lo}, {hi}] → set to NaN')
    else:
        print(f'  ✓  {feat:<15} all values within clinical range')

print(f'\n  Total values flagged and set to NaN: {total_flagged:,}')

Blood pressure features:
  Mean_SBP valid: 1,451
  Mean_DBP valid: 1,451
  Readings per participant:
    1 reading(s): 2
    2 reading(s): 1
    3 reading(s): 1,448
    0 readings:   152

eGFR (CKD-EPI 2021):
  Valid:  1,499
  Mean:   110.1 mL/min/1.73m²
  Range:  4.7–142.6 mL/min/1.73m²

Direct biomarkers assigned:
  HbA1c            1,522  (94.9%)
  BMI              1,596  (99.6%)
  Waist            1,552  (96.8%)
  HDL              1,503  (93.8%)
  Glucose          1,497  (93.4%)
  ALT              1,498  (93.4%)
  Triglycerides      745  (46.5%)
  LDL                744  (46.4%)

Clinical range validation:
  ✓  HbA1c           all values within clinical range
  ✓  BMI             all values within clinical range
  ✓  Waist           all values within clinical range
  ✓  Mean_SBP        all values within clinical range
  ✓  Mean_DBP        all values within clinical range
  ✓  eGFR            all values within clinical range
  ✓  HDL             all values within clinical range
  ✓ 

Clinical range validation flagged 3 values as physiologically implausible — 
all triglycerides below 20 mg/dL (values of 10, 16, and 18 mg/dL in women 
aged 38–40). Even in severe malnutrition or malabsorption, triglycerides 
rarely fall below 30 mg/dL — these values are consistent with measurement 
or data entry error and were set to NaN.

All other biomarker values, including extreme values that might appear 
suspicious at first glance, were retained after clinical review. In a 
nationally representative population survey, extreme values are more 
likely to represent rare but real clinical presentations — severe obesity, 
end-stage kidney disease, hyperalphalipoproteinemia — than measurement 
error. The clinical ranges applied here reflect physiological possibility 
rather than typical population distributions, deliberately avoiding 
over-exclusion of clinically meaningful outliers.

To illustrate this, five values that initially appeared to exceed the range limits were 
retained after clinical review:

- **BMI 82.0 and 92.3** (ages 37 and 34) — extreme but physiologically 
  real severe obesity. Upper limit raised from 80 to 100.
- **eGFR 4.7** (age 44, creatinine 9.48 mg/dL) — end-stage kidney disease. 
  Creatinine of 9.48 is severely elevated but clinically possible. Lower limit 
  reduced from 5.0 to 3.0.
- **HDL 151.0 and 159.0** (ages 43 and 42) — rare but physiologically 
  possible hyperalphalipoproteinemia. Upper limit raised from 150 to 200.


In [7]:
# ── Merge engineered CKM features onto main dataframe ─────────────────────
# ckm_features contains engineered variables (Mean_SBP, Mean_DBP, eGFR,
# and direct biomarkers) — merge onto df_ckm before medication flags
# WTSAFPRP dropped — already present in df_ckm from module integration

ckm_to_merge = ckm_features.drop(columns=['WTSAFPRP'])

n_before = len(df_ckm)
df_ckm   = df_ckm.join(ckm_to_merge, how='left')

assert len(df_ckm) == n_before, \
    f'Row count changed: {len(df_ckm)}'

print(f'✓ Engineered CKM features merged onto df_ckm')
print(f'  Shape: {df_ckm.shape}')
print(f'  Columns added: {list(ckm_to_merge.columns)}')

✓ Engineered CKM features merged onto df_ckm
  Shape: (1603, 53)
  Columns added: ['Mean_SBP', 'Mean_DBP', 'eGFR', 'HbA1c', 'BMI', 'Waist', 'HDL', 'Glucose', 'ALT', 'Triglycerides', 'LDL']


---
## Section 6 · Medication Flags from P_RXQ_RX

Women on CKM-relevant medications have biomarker values that reflect 
treatment rather than their underlying disease state. A woman on 
metformin will have lower HbA1c than her untreated baseline. A woman 
on antihypertensives will have lower blood pressure than her untreated 
state. If these women are clustered without flagging their medication 
use, their biomarker profiles will underestimate their true 
cardiometabolic risk — potentially placing them in lower-risk clusters 
than they belong in.

This section links the Prescription Medications module (P_RXQ_RX) to 
the analytical sample and creates binary flags for five medication 
categories with direct CKM relevance:

- **On_Metformin** — glucose-lowering agent, signals pre-diabetes, 
  type 2 diabetes, or PCOS treatment. Lowers HbA1c and glucose.
- **On_Antihypertensive** — blood pressure-lowering agent. Lowers 
  systolic and diastolic BP.
- **On_Statin** — lipid-lowering agent. Lowers LDL and triglycerides, 
  may raise HDL.
- **On_Insulin** — glucose-lowering agent, signals diabetes. Lowers 
  HbA1c and glucose — stronger effect than metformin.
- **On_Hormonal_Contraception** — modifies lipid profile, blood 
  pressure, and glucose metabolism. Relevant confounder for CKM 
  biomarker interpretation in reproductive-age women.

### P_RXQ_RX structure


P_RXQ_RX is structured differently from other NHANES modules — it is 
a long-format file where each row represents one prescription medication 
for one participant. A participant taking 3 medications will have 3 rows. 
The 2017–March 2020 release contains 32,962 medication records across 
15,560 participants (average 2.1 medications per participant).

Medications are identified by generic drug name in the `RXDDRUG` variable. 
The Multum Lexicon therapeutic category codes (RXDDCI1A–RXDDCI4B) 
referenced in earlier NHANES releases are not present in this file — 
drug name matching is therefore the primary identification method.

Drug name matching has one known limitation: 8,716 records (26.4%) have 
a blank drug name — participants who reported taking a medication but 
could not identify it. These unknown medications cannot be classified and 
represent a source of undercounting for all medication flags. This 
limitation is documented and accepted — the flags created here identify 
confirmed medication use, not all possible medication use.

In [8]:
# ── Section 6: Medication flags from P_RXQ_RX ─────────────────────────────
# P_RXQ_RX is long format — one row per medication per participant
# Must aggregate to one row per participant before merging

# ── Load and scan P_RXQ_RX ────────────────────────────────────────────────
rxq = pd.read_sas(RAW_DIR / 'P_RXQ_RX.xpt',
                  format='xport', encoding='utf-8')

print(f'P_RXQ_RX loaded:')
print(f'  Shape:         {rxq.shape}')
print(f'  Unique SEQNs:  {rxq["SEQN"].nunique():,}')
print(f'  Rows per SEQN: {rxq.shape[0] / rxq["SEQN"].nunique():.1f} average')

# Artifact scan
rxq_indexed  = rxq.set_index('SEQN')
rxq_artifacts = scan_sas_artifacts(rxq_indexed)
if rxq_artifacts:
    print(f'\n  ⚠️  Artifacts detected: {rxq_artifacts}')
    for col in rxq_artifacts:
        rxq_indexed[col] = np.where(
            rxq_indexed[col].notna() & (rxq_indexed[col].abs() < EPSILON),
            0.0, rxq_indexed[col]
        )
    print(f'  ✓ Fixed')
else:
    print(f'\n  ✓ No artifacts detected')

# ── Explore available columns ──────────────────────────────────────────────
print(f'\nP_RXQ_RX columns:')
print(rxq.columns.tolist())

# Check drug name and therapeutic category columns
print(f'\nSample drug names (RXDDRUG):')
drug_counts = rxq['RXDDRUG'].value_counts(dropna=False)

# Replace blank/empty string with readable label
drug_counts.index = drug_counts.index.where(
    drug_counts.index.str.strip() != '', 
    'UNKNOWN — participant could not name medication'
)
print(drug_counts.head(20))

P_RXQ_RX loaded:
  Shape:         (32962, 13)
  Unique SEQNs:  15,560
  Rows per SEQN: 2.1 average

  ✓ No artifacts detected

P_RXQ_RX columns:
['SEQN', 'RXDUSE', 'RXDDRUG', 'RXDDRGID', 'RXQSEEN', 'RXDDAYS', 'RXDRSC1', 'RXDRSC2', 'RXDRSC3', 'RXDRSD1', 'RXDRSD2', 'RXDRSD3', 'RXDCOUNT']

Sample drug names (RXDDRUG):
RXDDRUG
UNKNOWN — participant could not name medication    8716
ATORVASTATIN                                       1000
METFORMIN                                           896
LISINOPRIL                                          862
AMLODIPINE                                          787
METOPROLOL                                          676
LEVOTHYROXINE                                       674
OMEPRAZOLE                                          551
LOSARTAN                                            525
ALBUTEROL                                           520
SIMVASTATIN                                         480
GABAPENTIN                                          425
HYD

In [9]:
# ── Search for hormonal contraception and insulin ─────────────────────────
print('=== Searching for target drug classes ===\n')

# Search terms for each drug class
search_terms = {
    'Hormonal contraception': [
        'NORETHINDRONE', 'LEVONORGESTREL', 'DESOGESTREL', 'ETONOGESTREL',
        'NORGESTIMATE', 'DROSPIRENONE', 'MEDROXYPROGESTERONE',
        'ETHINYL ESTRADIOL', 'NEXPLANON', 'MIRENA', 'NUVARING',
        'ORTHO', 'SPRINTEC', 'LOESTRIN', 'SEASONIQUE', 'YASMIN',
        'YAZ', 'JUNEL', 'MICROGESTIN', 'CAMILA', 'ERRIN'
    ],
    'Insulin': [
        'INSULIN', 'GLARGINE', 'DETEMIR', 'ASPART', 'LISPRO',
        'DEGLUDEC', 'LANTUS', 'HUMALOG', 'NOVOLOG', 'LEVEMIR',
        'BASAGLAR', 'TOUJEO'
    ],
}

for category, terms in search_terms.items():
    print(f'{category}:')
    pattern = '|'.join(terms)
    matches = rxq[rxq['RXDDRUG'].str.contains(
        pattern, case=False, na=False
    )]
    if len(matches) > 0:
        print(matches['RXDDRUG'].value_counts().to_string())
    else:
        print('  No matches found')
    print()

=== Searching for target drug classes ===

Hormonal contraception:
RXDDRUG
ETHINYL ESTRADIOL; NORETHINDRONE                 83
ETHINYL ESTRADIOL; NORGESTIMATE                  76
ETHINYL ESTRADIOL; LEVONORGESTREL                29
ETONOGESTREL                                     16
DROSPIRENONE; ETHINYL ESTRADIOL                  15
NORETHINDRONE                                    13
MEDROXYPROGESTERONE                              13
ETHINYL ESTRADIOL; NORGESTREL                    12
ETHINYL ESTRADIOL; NORELGESTROMIN                10
DESOGESTREL; ETHINYL ESTRADIOL                   10
LEVONORGESTREL                                    9
ETHINYL ESTRADIOL; ETONOGESTREL                   7
ESTRADIOL; NORETHINDRONE                          3
CONJUGATED ESTROGENS; MEDROXYPROGESTERONE         2
ETHINYL ESTRADIOL; ETHYNODIOL                     2
ETHINYL ESTRADIOL; GESTODENE                      1
DROSPIRENONE; ETHINYL ESTRADIOL; LEVOMEFOLATE     1
ETHINYL ESTRADIOL                        

In [10]:
# ── Medication flag construction ───────────────────────────────────────────
# Match by generic drug name using keyword patterns
# All matches are case-insensitive
# Participants with unknown drug names (blank RXDDRUG) cannot be classified

# ── Define drug class patterns ─────────────────────────────────────────────
drug_patterns = {
    'On_Metformin': [
        'METFORMIN'
    ],
    'On_Statin': [
        'ATORVASTATIN', 'SIMVASTATIN', 'ROSUVASTATIN', 'PRAVASTATIN',
        'LOVASTATIN', 'FLUVASTATIN', 'PITAVASTATIN'
    ],
    'On_Antihypertensive': [
        # ACE inhibitors
        'LISINOPRIL', 'ENALAPRIL', 'RAMIPRIL', 'BENAZEPRIL',
        'CAPTOPRIL', 'FOSINOPRIL', 'QUINAPRIL', 'PERINDOPRIL',
        # ARBs
        'LOSARTAN', 'VALSARTAN', 'IRBESARTAN', 'OLMESARTAN',
        'CANDESARTAN', 'TELMISARTAN', 'AZILSARTAN',
        # Calcium channel blockers
        'AMLODIPINE', 'NIFEDIPINE', 'DILTIAZEM', 'VERAPAMIL',
        'FELODIPINE', 'NICARDIPINE',
        # Beta blockers
        'METOPROLOL', 'ATENOLOL', 'CARVEDILOL', 'BISOPROLOL',
        'NEBIVOLOL', 'PROPRANOLOL', 'LABETALOL',
        # Diuretics
        'HYDROCHLOROTHIAZIDE', 'CHLORTHALIDONE', 'FUROSEMIDE',
        'SPIRONOLACTONE', 'INDAPAMIDE', 'TORSEMIDE',
        # Other
        'CLONIDINE', 'HYDRALAZINE', 'MINOXIDIL', 'DOXAZOSIN',
        'TERAZOSIN', 'PRAZOSIN'
    ],
    'On_Insulin': [
        'INSULIN'
    ],
    'On_Hormonal_Contraception': [
        'ETHINYL ESTRADIOL', 'NORETHINDRONE', 'NORGESTIMATE',
        'LEVONORGESTREL', 'ETONOGESTREL', 'DROSPIRENONE',
        'MEDROXYPROGESTERONE', 'DESOGESTREL', 'NORELGESTROMIN',
        'NORGESTREL', 'GESTODENE', 'ETHYNODIOL'
    ],
}

# ── Restrict to analytical sample ─────────────────────────────────────────
analytical_seqn = set(df_ckm.index)
rxq_analytical  = rxq[rxq['SEQN'].isin(analytical_seqn)].copy()

print(f'P_RXQ_RX records for analytical sample:')
print(f'  Total records:     {len(rxq_analytical):,}')
print(f'  Unique participants: {rxq_analytical["SEQN"].nunique():,}')
print(f'  No medications:    {len(analytical_seqn) - rxq_analytical["SEQN"].nunique():,}')

# ── Build flags ────────────────────────────────────────────────────────────
print(f'\n=== Medication Flag Results ===\n')
print(f'  {"Flag":<25} {"N users":>8} {"% sample":>9}  {"Top drugs"}')
print(f'  {"─"*25} {"─"*8} {"─"*9}  {"─"*35}')

med_flags = pd.DataFrame(index=df_ckm.index)

for flag, patterns in drug_patterns.items():
    pattern   = '|'.join(patterns)
    
    # Find matching records
    matches   = rxq_analytical[
        rxq_analytical['RXDDRUG'].str.contains(
            pattern, case=False, na=False
        )
    ]
    
    # Get unique SEQNs with this medication
    users     = set(matches['SEQN'].unique())
    
    # Create binary flag — 1 if user, 0 if not, for all analytical sample
    med_flags[flag] = med_flags.index.map(
        lambda x: 1.0 if x in users else 0.0
    )
    
    n_users   = len(users)
    pct       = n_users / len(df_ckm) * 100
    
    # Top drug names for this class
    top_drugs = matches['RXDDRUG'].value_counts().head(3).index.tolist()
    top_str   = ', '.join([d[:20] for d in top_drugs])
    
    print(f'  {flag:<25} {n_users:>8,} {pct:>8.1f}%  {top_str}')

# ── Validation ─────────────────────────────────────────────────────────────
assert len(med_flags) == 1603, \
    f'Med flags row count wrong: {len(med_flags)}'
assert all(med_flags.notna().all()), \
    'NaN values in medication flags — expected 0 or 1 only'

print(f'\n✓ All medication flag assertions passed')
print(f'  Shape: {med_flags.shape}')
print(f'\nNote: Women with unknown drug names (blank RXDDRUG) cannot be')
print(f'classified — flags represent confirmed use, not all possible use')

P_RXQ_RX records for analytical sample:
  Total records:     2,474
  Unique participants: 1,603
  No medications:    0

=== Medication Flag Results ===

  Flag                       N users  % sample  Top drugs
  ───────────────────────── ──────── ─────────  ───────────────────────────────────
  On_Metformin                    48      3.0%  METFORMIN
  On_Statin                       22      1.4%  ATORVASTATIN, PRAVASTATIN, SIMVASTATIN
  On_Antihypertensive            116      7.2%  LISINOPRIL, LOSARTAN, AMLODIPINE
  On_Insulin                      16      1.0%  INSULIN ASPART, INSULIN GLARGINE, INSULIN DETEMIR
  On_Hormonal_Contraception      173     10.8%  ETHINYL ESTRADIOL; N, ETHINYL ESTRADIOL; N, ETHINYL ESTRADIOL; L

✓ All medication flag assertions passed
  Shape: (1603, 5)

Note: Women with unknown drug names (blank RXDDRUG) cannot be
classified — flags represent confirmed use, not all possible use


Before building medication flags, the file structure requires 
investigation — all 1,603 analytical sample women appear in P_RXQ_RX, 
which is unexpected if the file only contains medication records. 
The RXDUSE variable reveals why.

In [11]:
# Verify — how many women have zero prescription medications?
print('Medication records per analytical sample participant:')
rxq_counts = rxq_analytical.groupby('SEQN').size()
print(f'  Min medications: {rxq_counts.min():,}')
print(f'  Max medications: {rxq_counts.max():,}')
print(f'  Mean medications: {rxq_counts.mean():.1f}')
print(f'\nWomen with exactly 1 record:')
print(f'  {(rxq_counts == 1).sum():,}')
print(f'\nFirst few records for a woman with 1 medication:')
one_med_seqn = rxq_counts[rxq_counts == 1].index[0]
print(rxq_analytical[rxq_analytical['SEQN'] == one_med_seqn][
    ['SEQN', 'RXDUSE', 'RXDDRUG', 'RXDCOUNT']
])

Medication records per analytical sample participant:
  Min medications: 1
  Max medications: 13
  Mean medications: 1.5

Women with exactly 1 record:
  1,269

First few records for a woman with 1 medication:
       SEQN  RXDUSE RXDDRUG  RXDCOUNT
3  109266.0     2.0               NaN


Confirming that RXDUSE=2 women have exactly one record each — 
a single row documenting their negative response to prescription use.

In [12]:
# Check RXDUSE values — this is the key variable
print('RXDUSE value counts in analytical sample:')
print(rxq_analytical['RXDUSE'].value_counts(dropna=False))

print('\nRXDUSE codebook:')
print('  1 = Yes (used prescription medication in past 30 days)')
print('  2 = No  (did not use prescription medication)')
print('  7 = Refused')
print('  9 = Don\'t know')

print('\nHow many analytical sample women said No to prescription use:')
no_meds = rxq_analytical[rxq_analytical['RXDUSE'] == 2.0]
print(f'  {no_meds["SEQN"].nunique():,} women')

print('\nHow many said Yes:')
yes_meds = rxq_analytical[rxq_analytical['RXDUSE'] == 1.0]
print(f'  {yes_meds["SEQN"].nunique():,} women')

RXDUSE value counts in analytical sample:
RXDUSE
1.0    1541
2.0     933
Name: count, dtype: int64

RXDUSE codebook:
  1 = Yes (used prescription medication in past 30 days)
  2 = No  (did not use prescription medication)
  7 = Refused
  9 = Don't know

How many analytical sample women said No to prescription use:
  933 women

How many said Yes:
  670 women


With the file structure understood, the correct denominators for 
interpreting medication flags can be established — distinguishing 
prevalence in the full analytical sample from prevalence among 
prescription users only.

In [13]:
# Women who explicitly said No to prescription medications
no_rx_seqn = set(rxq_analytical.loc[
    rxq_analytical['RXDUSE'] == 2.0, 'SEQN'
].unique())

# Women who said Yes
yes_rx_seqn = set(rxq_analytical.loc[
    rxq_analytical['RXDUSE'] == 1.0, 'SEQN'
].unique())

med_flags['Any_Prescription'] = med_flags.index.map(
    lambda x: 1.0 if x in yes_rx_seqn else 0.0
)

print('Prescription medication use in analytical sample:')
print(f'  Any prescription (RXDUSE=1): {len(yes_rx_seqn):,} '
      f'({len(yes_rx_seqn)/1603*100:.1f}%)')
print(f'  No prescription (RXDUSE=2):  {len(no_rx_seqn):,} '
      f'({len(no_rx_seqn)/1603*100:.1f}%)')
print(f'\nOf the {len(yes_rx_seqn):,} women on any prescription:')
for flag in drug_patterns.keys():
    n = (med_flags[flag] == 1.0).sum()
    pct_of_users = n / len(yes_rx_seqn) * 100
    print(f'  {flag:<25} {n:>4,}  ({pct_of_users:.1f}% of prescription users)')

Prescription medication use in analytical sample:
  Any prescription (RXDUSE=1): 670 (41.8%)
  No prescription (RXDUSE=2):  933 (58.2%)

Of the 670 women on any prescription:
  On_Metformin                48  (7.2% of prescription users)
  On_Statin                   22  (3.3% of prescription users)
  On_Antihypertensive        116  (17.3% of prescription users)
  On_Insulin                  16  (2.4% of prescription users)
  On_Hormonal_Contraception  173  (25.8% of prescription users)


In [14]:
# ── Merge medication flags onto CKM feature matrix ─────────────────────────
# Check if flags already merged
med_flag_cols = list(drug_patterns.keys()) + ['Any_Prescription']
already_merged = all(col in df_ckm.columns for col in med_flag_cols)

if already_merged:
    print('Medication flags already present in df_ckm — skipping merge')
else:
    n_before = len(df_ckm)
    df_ckm   = df_ckm.join(med_flags, how='left')
    assert len(df_ckm) == n_before, \
        f'Row count changed after medication merge: {len(df_ckm)}'
    print(f'✓ Medication flags merged onto CKM feature matrix')

print(f'  Shape: {df_ckm.shape}')
print(f'\nMedication flag summary:')
for flag in med_flag_cols:
    n   = (df_ckm[flag] == 1.0).sum()
    pct = n / len(df_ckm) * 100
    print(f'  {flag:<25} {n:>5,}  ({pct:.1f}%)')

✓ Medication flags merged onto CKM feature matrix
  Shape: (1603, 59)

Medication flag summary:
  On_Metformin                 48  (3.0%)
  On_Statin                    22  (1.4%)
  On_Antihypertensive         116  (7.2%)
  On_Insulin                   16  (1.0%)
  On_Hormonal_Contraception   173  (10.8%)
  Any_Prescription            670  (41.8%)


In [15]:
# Check if therapeutic category codes provide any recovery
print('RXDRSC1 coverage among records with blank drug names:')
blank_mask = rxq['RXDDRUG'].str.strip() == ''
print(rxq.loc[blank_mask, ['RXDRSC1', 'RXDRSC2', 'RXDRSC3']].notna().sum())
print('\nRXDRSC1 value counts (blank drug name records):')
print(rxq.loc[blank_mask, 'RXDRSC1'].value_counts(dropna=False).head(10))

RXDRSC1 coverage among records with blank drug names:
RXDRSC1    8716
RXDRSC2    8716
RXDRSC3    8716
dtype: int64

RXDRSC1 value counts (blank drug name records):
RXDRSC1
    8716
Name: count, dtype: int64


### Section 6 summary

Six medication flags were successfully created and merged onto the CKM 
feature matrix. All 1,603 analytical sample women are retained — 
medication flags are binary covariates, not exclusion criteria.

| Flag | N (% sample) | Clinical relevance |
|---|---|---|
| On_Metformin | 48 (3.0%) | Lowers HbA1c and glucose |
| On_Statin | 22 (1.4%) | Lowers LDL, modifies lipid profile |
| On_Antihypertensive | 116 (7.2%) | Lowers systolic and diastolic BP |
| On_Insulin | 16 (1.0%) | Strongly lowers HbA1c and glucose |
| On_Hormonal_Contraception | 173 (10.8%) | Modifies lipids, BP, and glucose |
| Any_Prescription | 670 (41.8%) | On any prescription medication |

**Known limitation — unidentifiable medications:**
8,716 records (26.4% of all prescription records) have a blank drug 
name. The therapeutic category codes (RXDRSC1–3) are also blank for 
all these records, confirming that no classification information is 
available through any variable in the file. These participants remain 
in the analytical sample and receive a value of 0 for any medication 
flag their unidentified medication would have triggered. This introduces 
systematic undercounting — the true prevalence of each drug class is 
likely somewhat higher than reported. No recovery strategy exists and 
this limitation is accepted as inherent to self-reported prescription 
data.

---
## Section 7 · CKM Feature Matrix Assembly

All biomarker modules and medication flags have been integrated. This 
section assembles the final CKM feature matrix, documents the complete 
feature inventory, and formally classifies each CKM feature by its 
intended use in the downstream pipeline — consistent with the 
reproductive feature classification established in notebook 02.

### CKM clustering features

The following CKM biomarkers are designated as clustering features for 
Stage 1 (full sample, n=1,603). Selection criteria:

1. **Non-fasting** — available for the full analytical sample
2. **Coverage ≥ 90%** — sufficient data for clustering without 
   excessive imputation
3. **Direct CKM relevance** — measures a distinct cardiometabolic 
   or kidney dimension

| Feature | Coverage | Clinical dimension |
|---|---|---|
| HbA1c | 94.9% | Glycemic control — glucose dysregulation marker |
| BMI | 99.6% | Adiposity — metabolic risk proxy |
| Waist | 96.8% | Central adiposity — visceral fat marker |
| Mean_SBP | 90.5% | Vascular — systolic blood pressure |
| Mean_DBP | 90.5% | Vascular — diastolic blood pressure |
| eGFR | 93.5% | Kidney function — filtration rate |
| HDL | 93.8% | Lipid — cardioprotective cholesterol |
| Glucose | 93.4% | Glycemic — random glucose |

**Stage 2 features (fasting subsample, n≈745):**
Triglycerides and LDL — examined within clusters post-hoc.

**Medication flags — confounders, not clustering features:**
All five medication flags are retained for cluster characterization 
and sensitivity analysis in notebook 04. The decision to include or 
exclude medicated women from clustering is deferred to notebook 04.

In [16]:
# ── CKM feature matrix assembly ────────────────────────────────────────────
# Define complete feature inventory with intended use classification

CKM_FEATURE_CLASSIFICATION = {
    # ── Stage 1 clustering features — non-fasting, full sample ────────────
    'HbA1c':    'clustering',
    'BMI':      'clustering',
    'Waist':    'clustering',
    'Mean_SBP': 'clustering',
    'Mean_DBP': 'clustering',
    'eGFR':     'clustering',
    'HDL':      'clustering',
    'Glucose':  'clustering',

    # ── Stage 2 features — fasting subsample only ─────────────────────────
    'Triglycerides': 'stage2',
    'LDL':           'stage2',
    'WTSAFPRP':      'weight',   # fasting subsample survey weight

    # ── Medication flags — confounders ────────────────────────────────────
    'On_Metformin':              'confounder',
    'On_Statin':                 'confounder',
    'On_Antihypertensive':       'confounder',
    'On_Insulin':                'confounder',
    'On_Hormonal_Contraception': 'confounder',
    'Any_Prescription':          'confounder',
}

# ── Confirm all features present ──────────────────────────────────────────
missing = [f for f in CKM_FEATURE_CLASSIFICATION if f not in df_ckm.columns]
if missing:
    print(f'⚠️  Missing features: {missing}')
else:
    print(f'✓ All {len(CKM_FEATURE_CLASSIFICATION)} CKM features present')

# ── Feature summary table ──────────────────────────────────────────────────
print(f'\n{"═"*70}')
print(f'CKM FEATURE MATRIX SUMMARY')
print(f'{"═"*70}\n')

print(f'  {"Feature":<25} {"Valid":>6} {"NaN":>6} {"% Valid":>8} {"Use":>12}')
print(f'  {"─"*25} {"─"*6} {"─"*6} {"─"*8} {"─"*12}')

clustering_ckm = []
stage2_ckm     = []
confounder_ckm = []

for feat, use in CKM_FEATURE_CLASSIFICATION.items():
    if feat not in df_ckm.columns:
        continue
    n_valid = df_ckm[feat].notna().sum()
    n_nan   = df_ckm[feat].isna().sum()
    pct     = n_valid / len(df_ckm) * 100

    print(f'  {feat:<25} {n_valid:>6,} {n_nan:>6,} {pct:>7.1f}% {use:>12}')

    if use == 'clustering':
        clustering_ckm.append(feat)
    elif use == 'stage2':
        stage2_ckm.append(feat)
    elif use == 'confounder':
        confounder_ckm.append(feat)

print(f'\n{"─"*70}')
print(f'\nFeature counts by intended use:')
print(f'  Clustering (Stage 1):  {len(clustering_ckm):>3}  features')
print(f'  Stage 2 (fasting):     {len(stage2_ckm):>3}  features')
print(f'  Confounders:           {len(confounder_ckm):>3}  features')

# ── Combined clustering feature set ───────────────────────────────────────
print(f'\n{"─"*70}')
print(f'\nComplete Stage 1 clustering feature set:')
print(f'  Reproductive features (from notebook 02):')
repro_clustering = ['Age_Menarche', 'Parity', 'Pregnancy_Loss', 'APO_Score']  # Gravidity → Pregnancy_Loss
for f in repro_clustering:
    if f in df_ckm.columns:
        valid = df_ckm[f].notna().sum()
        print(f'    {f:<22} {valid:>6,} valid ({valid/len(df_ckm)*100:.1f}%)')

print(f'  CKM biomarker features:')
for f in clustering_ckm:
    valid = df_ckm[f].notna().sum()
    print(f'    {f:<22} {valid:>6,} valid ({valid/len(df_ckm)*100:.1f}%)')

total_clustering = len(repro_clustering) + len(clustering_ckm)
print(f'\n  Total Stage 1 clustering features: {total_clustering}')
print(f'  (4 reproductive + {len(clustering_ckm)} CKM biomarkers)')

# ── Validation ─────────────────────────────────────────────────────────────
assert len(df_ckm) == 1603, \
    f'Row count wrong: {len(df_ckm)}'
assert df_ckm.index.name == 'SEQN', \
    'SEQN not index'
assert all(f in df_ckm.columns for f in clustering_ckm), \
    'Some clustering features missing'
assert len(clustering_ckm) == 8, \
    f'Expected 8 CKM clustering features, got {len(clustering_ckm)}'

print(f'\n✓ All feature matrix assertions passed')

✓ All 17 CKM features present

══════════════════════════════════════════════════════════════════════
CKM FEATURE MATRIX SUMMARY
══════════════════════════════════════════════════════════════════════

  Feature                    Valid    NaN  % Valid          Use
  ───────────────────────── ────── ────── ──────── ────────────
  HbA1c                      1,522     81    94.9%   clustering
  BMI                        1,596      7    99.6%   clustering
  Waist                      1,552     51    96.8%   clustering
  Mean_SBP                   1,451    152    90.5%   clustering
  Mean_DBP                   1,451    152    90.5%   clustering
  eGFR                       1,499    104    93.5%   clustering
  HDL                        1,503    100    93.8%   clustering
  Glucose                    1,497    106    93.4%   clustering
  Triglycerides                742    861    46.3%       stage2
  LDL                          744    859    46.4%       stage2
  WTSAFPRP                     

The complete Stage 1 clustering feature set comprises 12 features — 
4 reproductive history features from notebook 02 and 8 CKM biomarkers 
from this notebook. All 8 CKM biomarkers exceed 90% coverage in the 
analytical sample, confirming the two-stage design is viable without 
restricting the clustering to the fasting subsample.

The 12-feature clustering space covers five distinct cardiometabolic 
dimensions:

- **Glycemic:** HbA1c, Glucose
- **Adiposity:** BMI, Waist circumference  
- **Vascular:** Mean_SBP, Mean_DBP
- **Kidney:** eGFR
- **Lipid:** HDL

Combined with the four reproductive features — menarche timing, 
pregnancy burden, delivery burden, and APO composite score — this 
feature space is designed to reveal whether adverse pregnancy history 
is associated with distinct early cardiometabolic dysregulation 
patterns in women aged 20–44.

---
## Section 8 · Export

The complete CKM feature matrix is exported to the processed data 
directory for use in notebook 04. The export includes all reproductive 
features, CKM biomarkers, medication flags, and P_DEMO variables.

The updated feature classification — now including both reproductive 
and CKM features — is also exported as a reference for notebook 05.

In [17]:
# ── Export complete CKM feature matrix ────────────────────────────────────
OUTPUT_PATH = Path('../data/processed/ckm_features.csv')

# ── File 1: Complete CKM feature matrix ───────────────────────────────────
df_ckm.to_csv(OUTPUT_PATH)

print(f'✓ CKM feature matrix exported')
print(f'  Path:  {OUTPUT_PATH}')
print(f'  Shape: {df_ckm.shape}')
print(f'  Size:  {OUTPUT_PATH.stat().st_size / 1e6:.1f} MB')

# ── File 2: Updated feature classification ─────────────────────────────────
# Combine reproductive and CKM feature classifications
repro_class = pd.read_csv(CLASS_PATH)

ckm_class = pd.DataFrame([
    {'feature': feat, 'intended_use': use,
     'n_valid': df_ckm[feat].notna().sum(),
     'pct_valid': round(df_ckm[feat].notna().sum() / len(df_ckm) * 100, 1),
     'clustering': use == 'clustering'}
    for feat, use in CKM_FEATURE_CLASSIFICATION.items()
    if feat in df_ckm.columns
])

# Combine both classification files
full_class = pd.concat([repro_class, ckm_class], ignore_index=True)

CLASS_OUTPUT = Path('../data/processed/feature_classification_full.csv')
full_class.to_csv(CLASS_OUTPUT, index=False)

print(f'\n✓ Full feature classification exported')
print(f'  Path:  {CLASS_OUTPUT}')
print(f'  Shape: {full_class.shape}')
print(f'  Total clustering features: '
      f'{full_class["clustering"].sum()}')

# ── Validate exports ───────────────────────────────────────────────────────
verify = pd.read_csv(OUTPUT_PATH, index_col='SEQN')
assert verify.shape == df_ckm.shape, \
    f'Readback shape mismatch: {verify.shape}'
assert verify.index.name == 'SEQN', \
    'SEQN not index in exported file'

print(f'\n✓ Export validation passed')
print(f'\n=== Notebook 03 outputs ===')
print(f'  ckm_features.csv              — primary input for notebook 04')
print(f'  feature_classification_full.csv — complete feature reference '
      f'for notebook 05')

✓ CKM feature matrix exported
  Path:  ../data/processed/ckm_features.csv
  Shape: (1603, 59)
  Size:  0.5 MB

✓ Full feature classification exported
  Path:  ../data/processed/feature_classification_full.csv
  Shape: (36, 5)
  Total clustering features: 12

✓ Export validation passed

=== Notebook 03 outputs ===
  ckm_features.csv              — primary input for notebook 04
  feature_classification_full.csv — complete feature reference for notebook 05


---
## Section 9 · Summary

This notebook integrated six CKM biomarker modules with the reproductive 
feature matrix from notebook 02, engineered derived features, identified 
medication confounders, and produced the complete feature matrix for 
downstream analysis.

### What this notebook established

**Module integration:**
Six NHANES biomarker modules linked to the 1,603-woman analytical sample 
via SEQN left join. All 1,603 women retained throughout — left join 
preserves the analytical anchor defined in notebook 01.

**SAS artifact scan:**
Two modules required artifact fixes before analysis — P_BIOPRO (28,417 
values) and P_TRIGLY (614 values). The mandatory pre-analysis scan 
established in notebook 01 caught these issues before they could 
propagate into downstream features.

**CKM features engineered:**
- Mean_SBP and Mean_DBP — averaged from up to 3 oscillometric readings
- eGFR — calculated using CKD-EPI 2021 race-free equation from serum 
  creatinine and age
- 8 direct biomarkers — HbA1c, BMI, waist, HDL, glucose, ALT, 
  triglycerides, LDL

**Clinical range validation:**
3 triglyceride values (10, 16, 18 mg/dL) flagged as physiologically 
implausible and set to NaN. Five extreme values — BMI 82 and 92, eGFR 
4.7, HDL 151 and 159 — retained after clinical review as rare but 
physiologically real presentations.

**Medication flags:**
Six binary flags created from P_RXQ_RX prescription data. 41.8% of 
the analytical sample reported prescription medication use. Hormonal 
contraception (10.8%) and antihypertensives (7.2%) are the most 
prevalent classes — both directly modify CKM biomarker values and 
must be accounted for in cluster interpretation.

### Complete Stage 1 clustering feature set (12 features)

| Domain | Features |
|---|---|
| Reproductive history | Age_Menarche, Parity, Pregnancy_Loss, APO_Score |
| Glycemic | HbA1c, Glucose |
| Adiposity | BMI, Waist circumference |
| Vascular | Mean_SBP, Mean_DBP |
| Kidney | eGFR |
| Lipid | HDL |

### Two-stage clustering design confirmed

All 8 CKM biomarkers exceed 90% coverage — Stage 1 clustering on the 
full 1,603-woman sample is viable. Fasting lipids (triglycerides, LDL) 
available for 46% of the sample — used for Stage 2 post-hoc cluster 
characterization only.

### Known limitations

- 8,716 prescription records (26.4%) have blank drug names and 
  unclassifiable therapeutic codes — medication flags undercount 
  true prevalence
- Clinical range validation removed 3 triglyceride values — 
  conservative approach retains physiologically extreme but 
  plausible values
- eGFR estimated from creatinine — not directly measured GFR
- Blood pressure from a single visit — may not reflect habitual BP
- All biomarkers from a single cross-sectional measurement

### Output files

| File | Shape | Description |
|---|---|---|
| `ckm_features.csv` | 1,603 × 58 | Primary input for notebook 04 |
| `feature_classification_full.csv` | 35 × 5 | Complete feature reference for notebook 05 |

### What notebook 04 will do

Notebook 04 performs exploratory analysis of the complete CKM feature 
matrix. Key tasks include:
- Descriptive statistics by APO status (GDM vs no GDM)
- Biomarker distributions by reproductive history subgroup
- Correlation analysis among clustering features
- Impact of medication use on biomarker distributions
- Decision on whether to exclude medicated women from clustering
- Missingness patterns and imputation strategy for notebook 05

---
*NHANES 2017–March 2020 Pre-Pandemic | CKM Integration*
*Analytical sample: 1,603 women aged 20–44*
*Stage 1 clustering features: 12 (4 reproductive + 8 CKM biomarkers)*